In [26]:
import os
import asyncio
from google.adk.agents.llm_agent import LlmAgent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.agents.parallel_agent import ParallelAgent
from google.adk.agents import BaseAgent, ParallelAgent, LlmAgent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.genai.types import ThinkingConfig
from google.genai import types
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool
import uuid
from pydantic import BaseModel, Field, field_validator, AliasChoices
from typing import List
from google import genai
from google.genai import types
from pathlib import Path
from typing import Optional, List
from google.adk.planners import BasePlanner, BuiltInPlanner, PlanReActPlanner
from google.genai.types import Content, Blob
import base64
import tempfile
import nest_asyncio
nest_asyncio.apply()
import json

In [ ]:
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = ""
os.environ["GOOGLE_CLOUD_LOCATION"] = ""  

In [28]:
INPUT_DOCUMENT_PATH = "docs/Hammond Lighting.pdf"

In [29]:

# pdf to base 64
def pdf_to_base64(pdf_path, output_txt_filepath):
    """
    Converts a PDF file to a Base64 encoded string.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The Base64 encoded string of the PDF, or None if an error occurs.
    """
    try:
        with open(pdf_path, "rb") as pdf_file:
            pdf_bytes = pdf_file.read()

        base64_string = base64.b64encode(pdf_bytes).decode("utf-8")

        with open(output_txt_filepath, "w") as txt_file:
            txt_file.write(base64_string)
            print(f"Successfully converted '{pdf_path}' to Base64 and saved to '{output_txt_filepath}'")
            return base64_string

    except FileNotFoundError:
        print(f"Error: PDF file not found at {pdf_path}")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


pdf_file_path = INPUT_DOCUMENT_PATH
base64_encoded_pdf = pdf_to_base64(pdf_file_path, "base64.txt")

if base64_encoded_pdf:
    print("PDF successfully converted to Base64.")


base64_pdf_str= "base64.txt"

with open(base64_pdf_str, "r") as f:
    base64_content = f.read()


Successfully converted 'docs/Hammond Lighting.pdf' to Base64 and saved to 'base64.txt'
PDF successfully converted to Base64.


In [30]:
class SummaryResult(BaseModel):
    num_pages: int
    lots_detected: bool
    items_detected: bool
    notes_detected: bool

In [31]:
from typing import Optional, List
from pydantic import BaseModel, Field, field_validator, AliasChoices

class Item(BaseModel):
    qty: Optional[float] = Field(
        default=None,
        description="Numeric quantity value (count, volume, amount, etc.)",
        validation_alias=AliasChoices("qty", "quantity", "amount", "quantity_ordered")
    )
    type: Optional[str] = Field(
        default=None,
        description="Item type/category/classification",
        validation_alias=AliasChoices("type", "category", "designation", "description")
    )
    mfr: Optional[str] = Field(
        default=None,
        description="Manufacturer/brand/supplier name",
        validation_alias=AliasChoices("mfr", "brand", "mfg", "manufacturer")
    )
    partNumber: Optional[str] = Field(
        default=None,
        description="Part number or item identifier",
        validation_alias=AliasChoices("part_number", "part", "item_number", "catalog_number", "cat_number", "fixture", "series",
                                      "item_id", "product", "material_description", "description", "fixture_series")
    )
    unitPrice: Optional[str] = Field(
        default=None,
        description="Unit price or rate per item as string preserving formatting",
        validation_alias=AliasChoices("unit_price", "unit_dollar", "net_unit_price", "rate", "price_each", "rate_per_item", "unit_sell")
    )
    um: Optional[str] = Field(
        default=None,
        description="Unit of measure (e.g., each, lb, hr)",
        validation_alias=AliasChoices("um", "uom", "uq")
    )
    extPrice: Optional[str] = Field(
        default=None,
        description="Extended or total price as string",
        validation_alias=AliasChoices("ext_price", "ext_dollar", "price", "sales_price", "amount", "cost", "ext_sell")
    )
    lineNote: Optional[str] = Field(
        default=None,
        description="Line item specific note or comment",
        validation_alias=AliasChoices("line_note", "line_comment", "prod_note", "product_note")
    )
    leadTime: Optional[str] = Field(
        default=None,
        description="Lead time or delivery timeframe",
        validation_alias=AliasChoices("lead_time", "shipping_time")
    )

    @field_validator("qty", mode="before")
    @classmethod
    def clean_qty(cls, v):
        try:
            return float(str(v).strip()) if v else None
        except Exception:
            return None

    @field_validator("type", "mfr", "partNumber", "unitPrice", "um", "extPrice", "lineNote", "leadTime", mode="before")
    @classmethod
    def clean_str_fields(cls, v):
        if not v or str(v).strip().lower() in {"n/a", "na", "null", "none", "tbd"}:
            return None
        return str(v).strip()

class LotItem(BaseModel):
    qty: Optional[float] = Field(
        default=None,
        description="Numeric quantity value (count, volume, amount, etc.)",
        validation_alias=AliasChoices("qty", "quantity", "amount", "quantity_ordered")
    )
    type: Optional[str] = Field(
        default=None,
        description="Item type/category/classification",
        validation_alias=AliasChoices("type", "category", "designation", "description")
    )
    mfr: Optional[str] = Field(
        default=None,
        description="Manufacturer/brand/supplier name",
        validation_alias=AliasChoices("mfr", "brand", "mfg", "manufacturer")
    )
    partNumber: Optional[str] = Field(
        default=None,
        description="Part number or item identifier",
        validation_alias=AliasChoices("part_number", "part", "item_number", "catalog_number", "cat_number", "fixture", "series",
                                      "item_id", "product", "material_description", "description", "fixture_series")
    )
    lineNote: Optional[str] = Field(
        default=None,
        description="Line item specific note or comment",
        validation_alias=AliasChoices("line_note", "line_comment", "prod_note", "product_note")
    )
    leadTime: Optional[str] = Field(
        default=None,
        description="Lead time or delivery timeframe",
        validation_alias=AliasChoices("lead_time", "shipping_time")
    )
    @field_validator("qty", mode="before")
    @classmethod
    def clean_qty(cls, v):
        try:
            return float(str(v).strip()) if v else None
        except Exception:
            return None

    @field_validator("type", "mfr", "partNumber", "lineNote", "leadTime", mode="before")
    @classmethod
    def clean_str_fields(cls, v):
        if not v or str(v).strip().lower() in {"n/a", "na", "null", "none", "tbd"}:
            return None
        return str(v).strip()

class LotIdentifier(BaseModel):
    value: Optional[str] = Field(
        default=None,
        description="Lot or group identifier",
        validation_alias=AliasChoices("lot_id", "lot_number", "group_id", "section", "category_group")
    )

    @field_validator("value", mode="before")
    @classmethod
    def clean_identifier(cls, v):
        if not v or str(v).strip().lower() in {"n/a", "na", "null", "none", "tbd"}:
            return None
        return str(v).strip()

class LotSubtotal(BaseModel):
    value: Optional[str] = Field(
        default=None,
        description="Subtotal amount for a lot as string",
        validation_alias=AliasChoices("subtotal", "sub_total", "total", "lot_total", "group_total", "grand_total")
    )

    @field_validator("value", mode="before")
    @classmethod
    def clean_subtotal(cls, v):
        if not v or str(v).strip().lower() in {"n/a", "na", "null", "none", "tbd"}:
            return None
        return str(v).strip()

class Lot(BaseModel):
    identifier: Optional[LotIdentifier] = None
    items: List[LotItem] = []
    subtotal: Optional[LotSubtotal] = None
    lot_notes: Optional[str] = None

    @field_validator('lot_notes', mode='before')
    @classmethod
    def clean_notes(cls, v):
        if not v or str(v).strip().lower() in {"n/a", "na", "null", "none", "tbd"}:
            return None
        return str(v).strip()

class Note(BaseModel):
    Note: Optional[str] = Field(
        default=None,
        description="General customer notes",
        validation_alias=AliasChoices("Note", "Quote Notes", "Notes", "Important Note", "Assumptions", "Technical Notes", "Special Note")
    )

    termsAndConditions: Optional[str] = Field(
        default=None,
        description="Terms and conditions of sale or special terms",
        validation_alias=AliasChoices("terms and conditions of sale", "Terms", "Purchase Terms", "Special Terms", "TERMS & CONDITIONS", "Standard Terms and Conditions")
    )

    mfgTerms: Optional[str] = Field(
        default=None,
        description="MFG Terms"
    )

    comments: Optional[str] = Field(
        default=None,
        description="comments",
        validation_alias=AliasChoices("comments","General Comments", "Special Conditions")
    )

    disclaimers: Optional[str] = Field(
        default=None,
        description="Disclaimers"
    )

    freightOrderTerms: Optional[str] = Field(
        default=None,
        description="Freight/Order Terms"
    )


    @field_validator("Note","termsAndConditions","mfgTerms","comments", "disclaimers", "freightOrderTerms", mode="before")
    @classmethod
    def clean_terms(cls, v):
        if not v or str(v).strip().lower() in {"n/a", "na", "null", "none", "tbd"}:
            return None
        return str(v).strip()


class ItemTable(BaseModel):
    items: List[Item] = []

class LotTable(BaseModel):
    lots: List[Lot] = []

class NotesTable(BaseModel):
    additionalinfo: Optional[Note] = None


In [32]:
# Get working directory
base_dir = Path.cwd()

# Load raw prompt from file
def load_prompt(path):
    with open(path, "r", encoding="utf-8") as f:
        content= f.read()
        return f'""" {content} """'

# Load prompts
summary_prompt=load_prompt('./prompts/summary_prompt.txt')
Lot_prompt=load_prompt('./prompts/lot_prompt.txt')
Item_prompt=load_prompt('./prompts/Units_prompt.txt')
Notes_prompt=load_prompt('./prompts/notes_prompt.txt')

In [33]:
# Step 1: Create a ThinkingConfig
thinking_config = ThinkingConfig(
    include_thoughts=False,   # Ask the model to include its thoughts in the response
    thinking_budget=2000,      # Limit the 'thinking' to 256 tokens (adjust as needed)
)
print("ThinkingConfig:", thinking_config)

# Step 2: Instantiate BuiltInPlanner
planner = BuiltInPlanner(
    thinking_config=thinking_config
)

ThinkingConfig: include_thoughts=False thinking_budget=2000


In [34]:
def create_summary_agent():
    return LlmAgent(
        name="summary_extraction_agent",
        model="gemini-2.5-pro",
        output_schema=SummaryResult,
        instruction=summary_prompt,
        planner=planner,
        output_key="summary",
        generate_content_config=types.GenerateContentConfig(
        temperature=0.0)
    )


def create_lots_agent():
    return LlmAgent(
        name="lot_extraction_agent",
        model="gemini-2.5-pro",
        instruction=Lot_prompt,
        planner=planner,
        output_schema=LotTable,
        output_key="lots",
        generate_content_config=types.GenerateContentConfig(
        temperature=0.0)
    )

def create_units_agent():
    return LlmAgent(
        name="unit_extraction_agent",
        model="gemini-2.5-pro",
        instruction=Item_prompt,
        planner=planner,
        output_schema=ItemTable,
        output_key="items",
        generate_content_config=types.GenerateContentConfig(
        temperature=0.0)
    )

def create_notes_agent():
    return LlmAgent(
        name="notes_extraction_agent",
        model="gemini-2.5-pro",
        instruction=Notes_prompt,
        planner=planner,
        output_schema=NotesTable,
        output_key="notes",
        generate_content_config=types.GenerateContentConfig(
        temperature=0.0)
    )

In [22]:
# async def summary_extraction():
#     session_service = InMemorySessionService()
#     APP_NAME = "summary_app"
#     USER_ID = "user"

#     session = await session_service.create_session(
#         app_name=APP_NAME,
#         user_id=USER_ID,
#         session_id=str(uuid.uuid4()),
#         state={}
#     )
#     runner = Runner(app_name= APP_NAME ,agent=summary_agent, session_service=session_service)
#     blob = Blob(data=base64.b64decode(base64_content), mime_type="application/pdf")
#     message = Content(
#         role="user",
#         parts=[
#             types.Part(inline_data=blob)
#         ]
#     )
#     events = runner.run_async(
#             user_id="user",
#             session_id=session.id,
#             new_message=message
#         )

#         # Process events to get the final response.
#     async for event in events:
#         if event.is_final_response():
#             return event.content.parts[0].text
#     return "No response received."


    
# summary= await summary_extraction()

In [23]:
# --- 3. Create the Conditional Router Agent ---
class ConditionalParallelRouter(BaseAgent):
    """
    Inspects the 'summary' and runs the required 
    specialized agents in parallel.
    """
    # The method signature MUST accept only one argument after self: the context.
    async def _run_async_impl(self, context: ToolContext) -> dict:
        # The 'context' parameter IS the tool_context.
        summary = context.state.get("summary")
        if not summary:
            return {"error": "Summary output not found in state."}

        summary_dict = summary.model_dump() if hasattr(summary, "model_dump") else summary
        
        agents_to_run = []
        # Use factory functions to get fresh agent instances
        if summary_dict.get("lots_detected"):
            agents_to_run.append(create_lots_agent())
        if summary_dict.get("items_detected"):
            agents_to_run.append(create_units_agent())
        if summary_dict.get("notes_detected"):
            agents_to_run.append(create_notes_agent())

        if not agents_to_run:
            return {"info": "No entities detected to extract."}

        tasks = [
            # Pass the same context down to the sub-agents.
            AgentTool(agent=agent).run_async(args={"request": "Extract your data."}, tool_context=context)
            for agent in agents_to_run
        ]
        
        results = await asyncio.gather(*tasks, return_exceptions=True)

        final_output = {}
        for agent, result in zip(agents_to_run, results):
            output_key = agent.output_key or agent.name
            if isinstance(result, Exception):
                final_output[output_key] = {"error": str(result)}
            else:
                final_output[output_key] = result
        
        context.state["extractions"] = final_output
        return final_output

# --- 4. Assemble the final Sequential Workflow ---
workflow_agent = SequentialAgent(
    name="workflow_agent",
    sub_agents=[
        create_summary_agent(),
        ConditionalParallelRouter(name="conditional_router") # Use the router here
    ]
)

Invalid config for agent summary_extraction_agent: output_schema cannot co-exist with agent transfer configurations. Setting disallow_transfer_to_parent=True, disallow_transfer_to_peers=True


In [24]:
async def process_pdf_workflow(pdf_bytes: bytes) -> dict:
    # Create services
    session_service = InMemorySessionService()
    artifact_service = InMemoryArtifactService()
    memory_service = InMemoryMemoryService()

    runner = Runner(
        app_name="QPM_doc_app",
        agent=workflow_agent,
        session_service=session_service,
        artifact_service=artifact_service,
        memory_service=memory_service,
    )

    # Create message with PDF
    document = types.Content(
        role="user",
        parts=[
            types.Part(text="PDF provided. Perform workflow."),
            types.Part(inline_data=types.Blob(
                data=pdf_bytes,
                mime_type="application/pdf"
            ))
        ]
    )

    # Store document in session state for tool access
    session = await session_service.create_session(
    app_name="QPM_doc_app",
    user_id="user",
    session_id=str(uuid.uuid4()),
    state={"document": document}   # <-- important
)
    # Run root agent, ensuring document is always passed
    events = []
    async for event in runner.run_async(
        user_id="user",
        session_id=session.id,
        new_message=document
    ):
        events.append(event)

    # Collect responses
    agent_responses = []
    for event in events:
        if event.content and event.content.parts:
            response_text = ''.join(part.text or '' for part in event.content.parts if getattr(part, "text", None))
            if response_text and event.author != "user":
                agent_responses.append({
                    "agent": event.author,
                    "response": response_text
                })

    full_state = session.state  # same dict mutated by tool_context.state
    summary = full_state.get("summary")
    extractions = full_state.get("extractions")

    return {
        "total_events": len(events),
        "agent_responses": agent_responses,
        "summary": summary,
        "extractions": extractions,
        "processing_complete": True
    }


**Simple Random generator custom agent**

In [ ]:
from google.adk.agents import BaseAgent, SequentialAgent
import random

class RandomNumberAgent(BaseAgent):
    async def run(self, **kwargs):
        number = random.randint(1, 100)
        print(f"Generated number: {number}")
        return {"number": number}

class EvenAgent(BaseAgent):
    async def run(self, number, **kwargs):
        print(f"EvenAgent received: {number}")
        return {"result": f"{number} is even!"}

class OddAgent(BaseAgent):
    async def run(self, number, **kwargs):
        print(f"OddAgent received: {number}")
        return {"result": f"{number} is odd!"}

In [ ]:
from google.adk.agents import BaseAgent
from pydantic import Field

class EvenOddWorkflowAgent(BaseAgent):
    random_agent: RandomNumberAgent = Field(default_factory=lambda: RandomNumberAgent(name="random_agent"))
    even_agent: EvenAgent = Field(default_factory=lambda: EvenAgent(name="even_agent"))
    odd_agent: OddAgent = Field(default_factory=lambda: OddAgent(name="odd_agent"))

    async def run(self, **kwargs):
        result = await self.random_agent.run()
        number = result["number"]
        if number % 2 == 0:
            return await self.even_agent.run(number=number)
        else:
            return await self.odd_agent.run(number=number)

In [ ]:
import asyncio

workflow_agent = EvenOddWorkflowAgent(name="workflow_agent")
result = await workflow_agent.run()
print(result)

Generated number: 30
EvenAgent received: 30
{'result': '30 is even!'}


**Custom Agent**

In [36]:

class WorkflowAgent(BaseAgent):
    summary_agent: BaseAgent = Field(default_factory=lambda: create_summary_agent())
    lot_agent: BaseAgent = Field(default_factory=lambda: create_lots_agent())
    units_agent: BaseAgent = Field(default_factory=lambda: create_units_agent())
    notes_agent: BaseAgent = Field(default_factory=lambda: create_notes_agent())

    async def run(self, base64_content, app_names=None, user_id="user"):
        # Step 1: Create session and message
        session_service = InMemorySessionService()
        
        if app_names is None:
            app_names = {
                "summary": "summary_app",
                "lots": "lots_app",
                "units": "units_app",
                "notes": "notes_app"
            }

        session = await session_service.create_session(
            app_name=app_names["summary"],
            user_id="user",
            session_id=str(uuid.uuid4()),
            state={}
        )
        blob = Blob(data=base64.b64decode(base64_content), mime_type="application/pdf")
        message = Content(
            role="user",
            parts=[types.Part(inline_data=blob)]
        )

        # Step 2: Run summary extraction
        summary_runner = Runner(app_name=app_names["summary"], agent=self.summary_agent, session_service=session_service)
        events = summary_runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=message
        )
        summary_result = None
        async for event in events:
            if event.is_final_response():
                summary_result = event.content.parts[0].text
                break

        # Step 3: Parse summary_result (assume it's a dict after parsing)
        import json
        try:
            summary = json.loads(summary_result)
        except Exception as e:
            summary = {}
            extraction_results["error"] = f"Summary parsing failed: {e}"

        # Step 4: Prepare parallel extraction tasks based on summary
        extraction_tasks = []
        extraction_results = {}

        async def run_agent(agent, app_name):
            session = await session_service.create_session(
                app_name=app_name,
                user_id=user_id,
                session_id=str(uuid.uuid4()),
                state={}
            )
            runner = Runner(app_name=app_name, agent=agent, session_service=session_service)
            events = runner.run_async(
                user_id=user_id,
                session_id=session.id,
                new_message=message
            )
            async for event in events:
                if event.is_final_response():
                    return event.content.parts[0].text
            return None

        if summary.get("lots_detected", False):
            extraction_tasks.append(run_agent(self.lot_agent, app_names["lots"]))
        if summary.get("items_detected", False):
            extraction_tasks.append(run_agent(self.units_agent, app_names["units"]))
        if summary.get("notes_detected", False):
            extraction_tasks.append(run_agent(self.notes_agent, app_names["notes"]))

        # Step 5: Run extraction agents in parallel
        results = await asyncio.gather(*extraction_tasks)
        keys = []
        if summary.get("lots_detected", False): keys.append("lots")
        if summary.get("items_detected", False): keys.append("units")
        if summary.get("notes_detected", False): keys.append("notes")
        extraction_results = {}
        for k, v in zip(keys, results):
            try:
                extraction_results[k] = json.loads(v)
            except Exception:
                extraction_results[k] = v  # fallback to raw string if not valid JSON
        
        return json.dumps({
            "summary": summary,
            "extraction_results": extraction_results
        }, indent=2)

# Usage example:
workflow_agent = WorkflowAgent(name="workflow_agent")
result = await workflow_agent.run(base64_content)
print(result)

Invalid config for agent summary_extraction_agent: output_schema cannot co-exist with agent transfer configurations. Setting disallow_transfer_to_parent=True, disallow_transfer_to_peers=True
Invalid config for agent lot_extraction_agent: output_schema cannot co-exist with agent transfer configurations. Setting disallow_transfer_to_parent=True, disallow_transfer_to_peers=True
Invalid config for agent unit_extraction_agent: output_schema cannot co-exist with agent transfer configurations. Setting disallow_transfer_to_parent=True, disallow_transfer_to_peers=True
Invalid config for agent notes_extraction_agent: output_schema cannot co-exist with agent transfer configurations. Setting disallow_transfer_to_parent=True, disallow_transfer_to_peers=True
/Users/Nidhishree.Sanam/Downloads/QPM_GEMINI_FEEDBACK/.venv/lib/python3.12/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You mi

{
  "summary": {
    "num_pages": 3,
    "lots_detected": true,
    "items_detected": true,
    "notes_detected": true
  },
  "extraction_results": {
    "lots": {
      "lots": [
        {
          "identifier": {
            "lot_id": "F5/F6"
          },
          "items": [
            {
              "qty": 1,
              "type": "F5/F6",
              "mfr": "EVERBRITE",
              "part_number": "***TYPE F5 & F6 ARE PLUS FREIGHT***ESTIMATE FREIGHT FOR QUANTITIES QUOTED ABOVE***",
              "line_note": null,
              "lead_time": null
            }
          ],
          "subtotal": {
            "subtotal": "$400.00"
          },
          "lot_notes": null
        },
        {
          "identifier": {
            "lot_id": "F2/F8"
          },
          "items": [
            {
              "qty": 1,
              "type": "F2/F8",
              "mfr": "COL",
              "part_number": "***COLUMBIA TYPES F2 AND F8 WILL BE PLUS FREIGHT, ESTIMATE ONLY***",
    